In [14]:
import pandas as pd
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# =====================================
# 1. Load Data
# =====================================

df = pd.read_csv("../data/processed/YRBS_Cleaned_and_Recoded.csv")
df["NoMilkDrinking"] = df["NoMilkDrinking"].astype(int)

# =====================================
# 2. Tukey Test
# =====================================

tukey = pairwise_tukeyhsd(
    endog=df["HowTallAreYouWithoutShoesInMeters"],
    groups=df["NoMilkDrinking"],
    alpha=0.05
)

# =====================================
# 3. Convert to DataFrame
# =====================================

tukey_df = pd.DataFrame(
    data=tukey._results_table.data[1:],
    columns=tukey._results_table.data[0]
)

# 4. Fix p-value display
# =====================================

def format_p(p):
    if p < 0.001:
        return "<0.001"
    else:
        return round(p, 4)

tukey_df["p-adj"] = tukey_df["p-adj"].apply(format_p)
# =====================================
# 4. Label mapping
# =====================================

label_map = {
    0: "0 = No milk drinking",
    1: "1 = Occasional milk drinking",
    2: "2 = Frequent milk drinking"
}

tukey_df["group1"] = tukey_df["group1"].astype(int).map(label_map)
tukey_df["group2"] = tukey_df["group2"].astype(int).map(label_map)

# =====================================
# 5. 🔥 這裡才是重點：改「最左邊 index」
# =====================================

tukey_df.index = tukey_df.index.map(lambda x: f"Comparison {x+1}")

# =====================================
# 6. Rename columns
# =====================================

tukey_df = tukey_df.rename(columns={
    "group1": "Group 1",
    "group2": "Group 2",
    "meandiff": "Mean Difference",
    "p-adj": "Adjusted p-value (Tukey HSD)",
    "lower": "Lower CI",
    "upper": "Upper CI",
    "reject": "Significant Difference"
})

# =====================================
# 7. Display
# =====================================

display(tukey_df)

,Group 1,Group 2,Mean Difference,Adjusted p-value (Tukey HSD),Lower CI,Upper CI,Significant Difference
Comparison 1,0 = No milk drinking,1 = Occasional milk drinking,0.0104,<0.001,0.0048,0.0161,True
Comparison 2,0 = No milk drinking,2 = Frequent milk drinking,0.0364,<0.001,0.0307,0.0421,True
Comparison 3,1 = Occasional milk drinking,2 = Frequent milk drinking,0.0259,<0.001,0.0212,0.0306,True


Interpretation:
Tukey HSD results showed that all pairwise group comparisons were statistically significant (p < 0.001), indicating that height differs across all milk consumption levels.